In [4]:
import pandas as pd
import numpy as np

folder = '/Users/alejandrogomez-paz/Desktop/UFC Project/1. data_scraping/' #if running this code change the folder to your local folder name

df_failed = pd.read_csv(folder + 'failed_fights.csv')
df_missing_failed = pd.read_csv(folder + 'missing_failed_fights.csv')
print(df_failed.head(3), df_missing_failed.head(1))

                                                 url  \
0  http://ufcstats.com/fight-details/a5c90086fb65...   
1  http://ufcstats.com/fight-details/a1db4c917777...   
2  http://ufcstats.com/fight-details/80cb366cd1e5...   

                                               error  
0                  'NoneType' object is not iterable  
1                  'NoneType' object is not iterable  
2  cannot access local variable 'winner' where it...                                                    url  \
0  http://ufcstats.com/fight-details/8bfcaa85e06d...   

                               error  
0  'NoneType' object is not iterable  


The Failed fights and Missing Failed fights were non-scrapable fights. Only two reasons show NoneType meaning no data is avaliable to scrape for the given fight OR no winner due to 'Overturned' fight or 'No Contest' Fight. Either way this data can be safely discarded

Now to clean the data from the four usable csvs:

In [10]:
# cleaning fight oneline stats
df1 = pd.read_csv(folder + 'fight_oneline_stats.csv')
df2 = pd.read_csv(folder + 'missing_fight_oneline_stats.csv')
df_fight_oneline = pd.concat([df1, df2], ignore_index=True)

df_fight_oneline.columns = df_fight_oneline.columns.str.replace(':', '').str.replace(' ', '_').str.lower() #reformat col names
df_fight_oneline['time_sec'] = ((pd.to_numeric(df_fight_oneline['time'].str.split(':', expand=True)[0], errors = 'coerce') * 60) +
                       pd.to_numeric(df_fight_oneline['time'].str.split(':', expand=True)[1], errors = 'coerce')).astype('Int64') #clean time col to secs
df_fight_oneline = df_fight_oneline.drop(columns = 'time')

split = df_fight_oneline['time_format'].str.extract(r'(\d+)\s*Rnd\s*\((\d+)-') #clean time format col into two cols with clean datatypes
df_fight_oneline['round_total'] = pd.to_numeric(split[0], errors='coerce').astype('Int64')
df_fight_oneline['round_time_sec'] = pd.to_numeric(split[1], errors='coerce').astype('Int64') * 60
df_fight_oneline = df_fight_oneline.drop(columns='time_format')



# cleaning round by round data
df3 = pd.read_csv(folder + 'fights_roundbyround.csv')
df4 = pd.read_csv(folder + 'missing_fights_roundbyround.csv')
df_rbr = pd.concat([df3, df4], ignore_index=True)
df_rbr['Td %_x'] = df_rbr['Td %_x'].replace('---', np.nan) #clean '---' with NaNs

for col in df_rbr.columns: #drop duplicate columns error in scraping
    if col.endswith('_y'):
        df_rbr = df_rbr.drop(columns = col)

columns = ['Sig. str._x', 'Total str._x', 'Td_x', 'Sig. str', #convert cols with two numbers as a string to two seperate cols
           'Head', 'Body', 'Leg', 'Distance', 'Clinch', 'Ground']
for col in columns:
    if col in df_rbr.columns:
        split = df_rbr[col].str.split(' of ', expand=True)
        df_rbr[col + '_landed'] = pd.to_numeric(split[0], errors = 'coerce',).astype('Int64')
        df_rbr[col + '_attempted'] = pd.to_numeric(split[1], errors = 'coerce',).astype('Int64')
df_rbr = df_rbr.drop(columns = columns)

df_rbr.columns = (df_rbr.columns.str.lower().str.replace('.', '_').str.replace(' ', '_') #clean up column names
                  .str.replace('%', 'pct').str.replace('_x', '').str.replace('__', '_'))
df_rbr = df_rbr.rename(columns={'rev_': 'rev'})

columns = ['sig_str_pct', 'td_pct']
for col in columns: #clean cols to integer datatype
    if col in df_rbr.columns:
        df_rbr[col] = pd.to_numeric(df_rbr[col].str.replace('%', ''), errors='coerce').astype('Int64')

df_rbr['ctrl_secs'] = ((pd.to_numeric(df_rbr['ctrl'].str.split(':', expand=True)[0], errors = 'coerce') * 60) +
                       pd.to_numeric(df_rbr['ctrl'].str.split(':', expand=True)[1], errors = 'coerce')).astype('Int64') #clean time col
df_rbr = df_rbr.drop(columns = 'ctrl')




# cleaning fighter fights
df_fighter_fights = pd.read_csv(folder + 'fighter_fights.csv')
df_fighter_fights = df_fighter_fights.dropna(subset = ['W/L']).reset_index(drop=True) #removed empty rows due to scraping error
df_fights = df_fighter_fights.drop(columns = ['Kd', 'Str', 'Td', 'Sub']) #removed aggregate data duplicate from round by round df

df_fights['stoppage_time_sec'] = ((pd.to_numeric(df_fights['Time'].str.split(':', expand=True)[0], errors = 'coerce') * 60) +
                       pd.to_numeric(df_fights['Time'].str.split(':', expand=True)[1], errors = 'coerce')).astype('Int64') #clean time col
df_fights = df_fights.drop(columns = 'Time')
df_fights['Round'] = df_fights['Round'].astype('Int64') #clean Round datatype

split = df_fights['Fighter'].str.split(r'\s*\n\s*', regex=True) #clean and split fighters col into two with no whitespace
df_fights['opponent_name'] = split.str[-1]
df_fights['Fighter_name'] = split.str[0]
df_fights = df_fights.drop(columns = 'Fighter')

split = df_fights['Method'].str.split(r'\s*\n\s*', regex=True) #clean and split Method col into two with no whitespace
df_fights['method_type'] = split.str[0]
df_fights['method_specific'] = split.str[-1]
df_fights = df_fights.drop(columns = 'Method')

for row in df_fights.index: #remove redundancies
    if df_fights.loc[row, 'method_specific'] == df_fights.loc[row, 'method_type']:
        df_fights.loc[row, 'method_specific'] = np.nan

df_fights['Event'] = df_fights['Event'].str.split(r'\s*\n\s*', regex=True).str[0] #clean Events col whitespace formatting
df_fights = df_fights.rename(columns={'W/L': 'result'})
df_fights.columns = df_fights.columns.str.lower()




# cleaning fighter stats
df_fighter_stats = pd.read_csv(folder + 'fighter_stats.csv')
df_fighter_stats = df_fighter_stats.replace('--', np.nan) #clean '--' with NaNs

split = df_fighter_stats['record'].str.split('-', expand=True) #split record col into wins/losses/draws
df_fighter_stats['wins'] = pd.to_numeric(split[0], errors = 'coerce').astype('Int64')
df_fighter_stats['losses'] = pd.to_numeric(split[1], errors = 'coerce').astype('Int64')
df_fighter_stats['draws'] = pd.to_numeric(split[2], errors = 'coerce').astype('Int64')
df_fighter_stats = df_fighter_stats.drop(columns = 'record')

columns = ['Str. Acc.:', 'Str. Def:', 'TD Acc.:', 'TD Def.:']
for col in columns:
    new_col = col.replace(':', '').replace(' ', '_').rstrip('_') + '_pct'
    df_fighter_stats[new_col] = pd.to_numeric(df_fighter_stats[col].str.rstrip('%'), errors='coerce').astype('Int64')
df_fighter_stats.drop(columns=columns, inplace=True)

split = df_fighter_stats['Height:'].str.split("' ", expand=True) #convert height to inches
df_fighter_stats['height_inches'] = ((pd.to_numeric(split[0], errors = 'coerce') * 12) +
                                 pd.to_numeric(split[1].str.rstrip('"'), errors = 'coerce')).astype('Int64')
df_fighter_stats = df_fighter_stats.drop(columns = 'Height:')

df_fighter_stats.columns = (df_fighter_stats.columns.str.lower(). #reformating column names
    str.replace(' ', '_').str.replace('.', '').str.replace(':', ''))

df_fighter_stats['weight'] = pd.to_numeric(df_fighter_stats['weight'].str.replace(' lbs.', '', regex=False), errors='coerce').astype('Int64')
df_fighter_stats['reach'] = pd.to_numeric(df_fighter_stats['reach'].str.rstrip('"'), errors='coerce').astype('Int64')
name_map = df_fights[['fighter_id', 'fighter_name']].drop_duplicates()
df_fighter_stats = df_fighter_stats.merge(name_map, on='fighter_id', how='left')
df_fighter_stats = df_fighter_stats.drop(columns = 'unnamed_6')

Combine similar data to one table:

In [ ]:
df_fighters_fights = pd.merge(
    df_fight_oneline,
    df_fights,
    how='left',
    on= 'fight_id')

df_fighters_fights = df_fighters_fights.drop(columns = ['method', 'round_y', 'time_sec', 'result'])
df_fighters_fights = df_fighters_fights.rename(columns={'round_x': 'round_finished', 'winner': 'winner_name', 'loser': 'loser_name'})

df_fighters_fights['winner_id'] = None
df_fighters_fights['loser_id'] = None

for i, row in df_fighters_fights.iterrows():
    if row['fighter_name'] == row['winner_name']:
            df_fighters_fights.at[i, 'winner_id'] = row['fighter_id']
    else:
          df_fighters_fights.at[i, 'loser_id'] = row['fighter_id']

df_fighters_fights = df_fighters_fights.drop(columns = ['fighter_name', 'opponent_name', 'fighter_id'])
df_fighters_fights = df_fighters_fights.groupby('fight_id', as_index=False).first()



"\nfights Aggregate missing data to round = 0 (fight aggregate data)\n\n\ndf_rbr.to_csv('fights.csv', index=False)\ndf_fighters_fights.to_csv('fighters_fights.csv', index=False)\ndf_fighter_stats.to_csv('fighters.csv', index=False)\n"

fix fight meta-aggregate 'row 0''s missing data:

In [ ]:
stat_cols = ['sig_str_landed', 'sig_str_attempted', 'head_landed', 'head_attempted',
             'body_landed', 'body_attempted', 'leg_landed', 'leg_attempted',
             'distance_landed', 'distance_attempted', 'clinch_landed', 'clinch_attempted',
             'ground_landed', 'ground_attempted']

totals = (df_rbr[df_rbr['round'] > 0]
          .groupby(['fight_id', 'fighter'])[stat_cols]
          .sum())

for i, row in df_rbr.iterrows():
    if row['round'] == 0:
        for col in stat_cols:
            df_rbr.at[i, col] = totals.loc[(row['fight_id'], row['fighter']), col]

df_rbr.head()

,fight_id,fighter,kd,sig_str_pct,td_pct,sub_att,rev,round,sig_str_landed,sig_str_attempted,...,body_attempted,leg_landed,leg_attempted,distance_landed,distance_attempted,clinch_landed,clinch_attempted,ground_landed,ground_attempted,ctrl_secs
0,e761c5009c09b295,Alexandre Pantoja,0,61,42,1,0,0,32,52,...,5,12,14,30,50,2,2,0,0,166
1,e761c5009c09b295,Alexandre Pantoja,0,63,50,0,0,1,31,49,...,5,12,14,29,47,2,2,0,0,60
2,e761c5009c09b295,Alexandre Pantoja,0,33,33,1,0,2,1,3,...,0,0,0,1,3,0,0,0,0,106
3,e761c5009c09b295,Kai Asakura,0,58,<NA>,0,0,0,17,29,...,6,2,3,17,28,0,1,0,0,24
4,e761c5009c09b295,Kai Asakura,0,57,<NA>,0,0,1,15,26,...,6,2,3,15,25,0,1,0,0,24


Save cleaned dataframes to csvs:

In [38]:
df_rbr.to_csv('fights.csv', index=False)
df_fighters_fights.to_csv('fighters_fights.csv', index=False)
df_fighter_stats.to_csv('fighters.csv', index=False)